In [1]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel
from langchain_core.messages import SystemMessage,HumanMessage

In [2]:
import asyncio
from typing import Optional

In [3]:
def get_llm():
    return OllamaLLM(
        model="tinyllama:1.1b",
        temperature=0
    )

llm=get_llm()

In [7]:
def run_reflection_loop():
    """
    Demonstrates a multi-step AI reflections loop to progressively improve
    a python function.
    """
    # --- The Core Task ---
    
    task_prompt= """
    Your task is to create a Python function named
    `calculate_factorial`.
    This function should do the following:
    1. Accept a single integer `n` as input.
    2. Calculate its factorial (n!).
    3. Include a clear docstring explaining what the functions does.
    4. Handle edge cases: The factorial of 0 is 1.
    5. Handle invalid input: Raise a ValueError if the input is a negative number.
    """
    
    # --- The Reflection Loop ---
    
    max_iterations=3
    current_code=""
    # we will build a conversation history to provide context in each step.
    message_history=[HumanMessage(content=task_prompt)]
    
    for i in range(max_iterations):
        print("\n" + "="*25 + f"REFLECTION LOOP: ITERATION {i+1}" + "="*25)
        # --- 1. GENERATE / REFINE STAGE ---
        # In the first iteration, it generates, In subsequent iterations, it refines.
        
        if i==0:
            print("\n>>> STAGE 1: GENERATING initial code ...")
            #the first message is just the task prompt.
            response=llm.invoke(message_history)
            current_code=response
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            # The message history now contains the task,
            # the last code and the last critique.
            # we instruct the model to apply the critique.
            
            message_history.append(HumanMessage(content='Please refine the code using the critiques provided'))
            response=llm.invoke(message_history)
            current_code=response
            
        print("\n --- Generated Code (v" + str (i+1) + ")  ---\n" + current_code)
        
        message_history.append(response) #Add the generated code to history
        
        # --- 2. REFLECT STAGE ---
        print("\n>>> STAGE 2: REFLECTING on the generated code ....")
        
        # create a specific prompt for the reflector agent.
        # this asks the model to act as a senior code reviewer.
        
        reflector_prompt =[
            SystemMessage(content= """
                          You are a senior software engineer and an expert in Python.
                          Your role is to perform a meticulous code review.
                          Crtically evaluate the provided Python code based on the original task requirements.
                          Looks for bugs, styles issues, missing edge cases, and areas for improvement.
                          if the code is perfect and meets all requirements,
                          response with the single phrase 'CODE_IS_PERFECT'.
                          otherwise, provide a bulleted list of your critiques.
                          
                          """),
            HumanMessage(content=f"original Task:\n {task_prompt}\n\n Code to Review:\n {current_code}")
            
        ]
        critique_response = llm.invoke(reflector_prompt)
        critique=critique_response
        
        
        # --- 3. STOPPING CONDITION ---
        
        if 'CODE_IS_PERFECT' in critique:
            print("\n --- Critique ---\n No further critiques found. The code is satisfactory")
            break
        print("\n --- Critique ---\n" + critique)
        
        # Add the critique to the history for the next refinement loop.
        
        message_history.append(HumanMessage(content=f'Critique of the previous code:\n{critique}'))
        
        print("\n" + "="*30 + "FINAL RESULT" + "+"*30)
        print("\n Final refined code after the reflection process:\n")
        print(current_code)
        

if __name__=="__main__":
    run_reflection_loop()
        


=========================REFLECTION LOOP: ITERATION 1=========================

>>> STAGE 1: GENERATING initial code ...

 --- Generated Code (v1)  ---
AI: Sure, here's an implementation of your function in Python:

```python
def calculate_factorial(n):
    """Calculates the factorial of a given integer n."""
    if n < 0:
        raise ValueError("Input must be non-negative.")
    
    if n == 1:
        return 1
    
    else:
        return n * calculate_factorial(n - 1)
```

Here's how to use the function:

```python
>>> calculate_factorial(5)
120
```

The function takes a single integer `n` as input and calculates its factorial (i.e., the product of all integers from 1 up to n). It handles edge cases by checking if the input is negative, raising a ValueError if it is. The docstring explains what the function does.

>>> STAGE 2: REFLECTING on the generated code ....

 --- Critique ---
AI: Yes, that's correct. Here's another implementation with comments explaining each step of the 